# Notebook 03i — Hurdle Model + TabPFN (target: TUMOR)

**Objetivo**: probar dos modelos avanzados con los mismos datos que 03c para comparar con el mejor lineal (Lasso R²=0.085).  
**Target**: `log1p(Tumor_Telomere fusion rate)` — mismas features que 03c (verdes + azules + mesenquimal).  
**Modelos**:  
- **Hurdle model**: modelo en dos etapas — primero predice si hay fusiones (cero vs no-cero), luego predice la magnitud.  
- **TabPFN**: modelo preentrenado para datos tabulares pequeños (Hollmann et al. 2025).  
**Referencia**: 03c Lasso R²=0.085.

## 1. Imports

In [ ]:
import os


os.environ["TABPFN_TOKEN"] = "TU_TOKEN_DE_TABPFN"

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from tabpfn import TabPFNRegressor

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f'CWD: {os.getcwd()}')
print('Librerías cargadas OK')

## 2. Carga de datos (idéntica a 03c)

In [ ]:
TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'

df = pd.read_csv('datos_limpios_modelo_a.csv')

df_verde = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                         usecols=['Patient_ID_IF_mod', 'firstResponse', 'treatment'])
df = df.merge(df_verde, on='Patient_ID_IF_mod', how='left')

df_azul = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                        usecols=['Patient_ID_IF_mod', 'biopsySite', 'hasSystemicPreTreatment',
                                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1'])
df = df.merge(df_azul, on='Patient_ID_IF_mod', how='left')

df_mes = pd.read_excel('Tumor_mesenchimal.xlsx')
df_mes.columns = ['cancer_type_code', 'Mesenchimal_origin']
df = df.merge(df_mes, on='cancer_type_code', how='left')

print(f'Shape final: {df.shape}')

## 3. Target

In [ ]:
y     = np.log1p(df[TARGET])           # target continuo para regresión
y_bin = (df[TARGET] > 0).astype(int)   # target binario para la etapa 1 del hurdle

print(f'Skewness log1p: {y.skew():.2f}')
print(f'% pacientes con fusiones > 0: {y_bin.mean()*100:.1f}%  ({y_bin.sum()} de {len(y_bin)})')

## 4. Features (idénticas a 03c)

In [ ]:
df['firstResponse'] = df['firstResponse'].fillna('Sin_dato')
df['Mesenchimal_origin'] = df['Mesenchimal_origin'].fillna('Sin_dato')
top15 = df['treatment'].value_counts().head(15).index.tolist()
df['treatment_agrupado'] = df['treatment'].apply(
    lambda x: x if x in top15 else ('Sin_dato' if pd.isna(x) else 'Otro')
)
df['hasSystemicPreTreatment'] = df['hasSystemicPreTreatment'].fillna('Sin_dato')
df['hasRadiotherapyPreTreatment'] = df['hasRadiotherapyPreTreatment'].fillna('Sin_dato')
df['treatmentGiven'] = df['treatmentGiven'].fillna('Sin_dato')
top10_biopsy = df['biopsySite'].value_counts().head(10).index.tolist()
df['biopsySite_agrupado'] = df['biopsySite'].apply(
    lambda x: x if x in top10_biopsy else ('Sin_dato' if pd.isna(x) else 'Otro')
)
top10_t1 = df['treatment1'].value_counts().head(10).index.tolist()
df['treatment1_agrupado'] = df['treatment1'].apply(
    lambda x: x if x in top10_t1 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

cols_features = ['firstResponse', 'treatment_agrupado', 'Mesenchimal_origin',
                 'cancer_type', 'biopsySite_agrupado', 'hasSystemicPreTreatment',
                 'hasRadiotherapyPreTreatment', 'treatmentGiven', 'treatment1_agrupado']

X = pd.get_dummies(df[cols_features], drop_first=True)
print(f'Shape X: {X.shape}')

## 5. Split (mismo random_state=42 que 03c para comparación justa)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Target binario alineado con el split
y_bin_train = y_bin.loc[X_train.index]
y_bin_test  = y_bin.loc[X_test.index]

# Escalado para el hurdle model (LogisticRegression + Lasso lo necesitan)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'% no-cero — train: {y_bin_train.mean()*100:.1f}%  |  test: {y_bin_test.mean()*100:.1f}%')

## 6. Hurdle Model

**Etapa 1** — LogisticRegression: predice si el paciente tiene fusiones (>0) o no.  
**Etapa 2** — Lasso: predice la magnitud, entrenado solo con los pacientes que SÍ tienen fusiones.  
**Predicción final** — si el clasificador dice "hay fusiones", usamos el Lasso; si no, predecimos 0.

In [ ]:
# --- Etapa 1: clasificador binario ---
clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_train_sc, y_bin_train)

acc = (clf.predict(X_test_sc) == y_bin_test).mean()
print(f'Etapa 1 — Accuracy clasificador: {acc*100:.1f}%')

# --- Etapa 2: regresor solo en no-ceros ---
mask_train = y_bin_train.values == 1
X_train_pos = X_train_sc[mask_train]
y_train_pos = y_train.values[mask_train]

reg = Lasso(alpha=0.001, max_iter=10000, random_state=42)
reg.fit(X_train_pos, y_train_pos)
print(f'Etapa 2 — Lasso entrenado con {mask_train.sum()} pacientes (no-ceros)')

# --- Predicción combinada ---
proba_test   = clf.predict_proba(X_test_sc)[:, 1]   # P(fusiones > 0)
reg_pred     = np.maximum(reg.predict(X_test_sc), 0) # sin negativos
y_pred_hurdle = np.where(proba_test >= 0.5, reg_pred, 0.0)

# Deshacer log1p → métricas en espacio original
y_pred_hurdle_orig = np.expm1(y_pred_hurdle)
y_test_orig        = np.expm1(np.array(y_test))

r2_hurdle   = r2_score(y_test_orig, y_pred_hurdle_orig)
mae_hurdle  = mean_absolute_error(y_test_orig, y_pred_hurdle_orig)
rmse_hurdle = np.sqrt(mean_squared_error(y_test_orig, y_pred_hurdle_orig))

print()
print('=== HURDLE MODEL ===')
print(f'  R²  (test, espacio original):  {r2_hurdle:.4f}')
print(f'  MAE  (espacio original):        {mae_hurdle:.4f}')
print(f'  RMSE (espacio original):        {rmse_hurdle:.4f}')

## 7. TabPFN

Modelo preentrenado de Hollmann et al. 2025. No necesita escalado.
Disenado para datasets pequenos (< 1000 muestras): se usa un subsample aleatorio del train (n=1000, random_state=42).

In [ ]:
print("Entrenando TabPFN (puede tardar unos segundos)...")

# TabPFN esta disenado para datasets pequenos (< 1000 muestras - Hollmann et al. 2025)
# Usamos subsample aleatorio de 1000 pacientes del train para respetar su regimen de diseno
rng = np.random.RandomState(42)
idx = rng.choice(len(X_train), size=1000, replace=False)
X_train_sub = X_train.iloc[idx]
y_train_sub = y_train.iloc[idx]

tabpfn = TabPFNRegressor(ignore_pretraining_limits=True)
tabpfn.fit(X_train_sub, y_train_sub)
y_pred_tabpfn = tabpfn.predict(X_test)

# Deshacer log1p → métricas en espacio original
y_pred_tabpfn_orig = np.expm1(y_pred_tabpfn)

r2_tabpfn   = r2_score(y_test_orig, y_pred_tabpfn_orig)
mae_tabpfn  = mean_absolute_error(y_test_orig, y_pred_tabpfn_orig)
rmse_tabpfn = np.sqrt(mean_squared_error(y_test_orig, y_pred_tabpfn_orig))

print()
print("=== TabPFN (subsample n=1000, condicion de diseno Hollmann et al. 2025) ===")
print(f"  R2  (test, espacio original):  {r2_tabpfn:.4f}")
print(f"  MAE  (espacio original):        {mae_tabpfn:.4f}")
print(f"  RMSE (espacio original):        {rmse_tabpfn:.4f}")

## 7b. Estabilidad del TabPFN — validacion del subsample

Como TabPFN se entrena con solo 1000 muestras aleatorias, hay que comprobar que el resultado
no depende de cuales 1000 pacientes se eligieron. Repetimos el experimento con 5 semillas distintas:
si el R2 es estable, el modelo aprende senal real; si varia mucho, el resultado es fruto del azar del subsample.

In [ ]:
seeds = [42, 0, 1, 7, 99]
r2_seeds = []
for seed in seeds:
    rng_s = np.random.RandomState(seed)
    idx_s = rng_s.choice(len(X_train), size=1000, replace=False)
    m = TabPFNRegressor(ignore_pretraining_limits=True)
    m.fit(X_train.iloc[idx_s], y_train.iloc[idx_s])
    r2_s = r2_score(np.expm1(np.array(y_test)), np.expm1(m.predict(X_test)))
    r2_seeds.append(r2_s)
    print(f"  seed={seed:>3d}  R2={r2_s:.4f}")

print()
print("=== Estabilidad TabPFN (n=1000, 5 semillas) ===")
print(f"  Media: {np.mean(r2_seeds):.4f} +/- {np.std(r2_seeds):.4f}")
print(f"  Min:   {np.min(r2_seeds):.4f}")
print(f"  Max:   {np.max(r2_seeds):.4f}")
print()
print("Referencia Lasso (n=3628): R2=0.0846")

## 8. Comparativa final

In [ ]:
print('=== COMPARATIVA — modelos avanzados vs Lasso (referencia 03c) ===')
print()
print(f'  {"Modelo":<35}  {"R² test":>8}  {"MAE":>8}  {"RMSE":>8}')
print(f'  {"-"*35}  {"-"*8}  {"-"*8}  {"-"*8}')
print(f'  {"Lasso (03c, referencia)":<35}  {0.0846:>8.4f}  {0.0542:>8.4f}  {0.1126:>8.4f}')
print(f'  {"Hurdle (Logistic + Lasso)":<35}  {r2_hurdle:>8.4f}  {mae_hurdle:>8.4f}  {rmse_hurdle:>8.4f}')
print(f'  {"TabPFN":<35}  {r2_tabpfn:>8.4f}  {mae_tabpfn:>8.4f}  {rmse_tabpfn:>8.4f}')
print()
modelos = {'Lasso (referencia)': 0.0846, 'Hurdle': r2_hurdle, 'TabPFN': r2_tabpfn}
ganador = max(modelos, key=modelos.get)
print(f'Mejor R²: {ganador}  ({modelos[ganador]:.4f})')

## 9. Gráficos real vs predicho

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (nombre, y_pred, color) in zip(axes, [
    ('Hurdle Model', y_pred_hurdle, 'mediumseagreen'),
    ('TabPFN',       y_pred_tabpfn, 'darkorange'),
]):
    y_pred_plot = np.expm1(np.array(y_pred))
    y_real = np.expm1(np.array(y_test))
    ax.scatter(y_real, y_pred_plot, alpha=0.3, s=10, color=color)
    lim = max(y_real.max(), y_pred_plot.max()) if y_pred_plot.max() > 0 else y_real.max()
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1)
    ax.set_xlabel('Real (espacio original, tumor)')
    ax.set_ylabel('Predicho (espacio original, tumor)')
    r2 = r2_hurdle if nombre == 'Hurdle Model' else r2_tabpfn
    ax.set_title(f'{nombre}
R²={r2:.4f}  |  ref. Lasso=0.0846')

plt.suptitle('Hurdle Model + TabPFN — TARGET: TUMOR', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/img/img_03i_hurdle_tabpfn.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img/img_03i_hurdle_tabpfn.png')

## 10. Validacion por Cross-Validation (5-fold) — Hurdle Model

El Hurdle es un modelo en dos etapas que no es un estimador sklearn estandar,
por lo que `cross_val_score` no funciona directamente con el.
Se aplica un loop manual de KFold: en cada vuelta se escala DENTRO del fold (sin contaminacion),
se entrena el clasificador y el regresor, y se combinan las predicciones.

TabPFN ya tiene un analisis equivalente en la seccion 7b (5 semillas distintas de subsample).

Referencia (un solo split seed=42): Hurdle R2=0.0484 | Lasso R2=0.0846
Referencia CV (03c): Lasso media=0.0554 +/- 0.0282

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# --- CV manual para Hurdle (no es estimador sklearn estandar) ---
r2_hurdle_cv = []
for train_idx, test_idx in kf.split(X):
    X_f_tr, X_f_te = X.iloc[train_idx], X.iloc[test_idx]
    y_f_tr, y_f_te = y.iloc[train_idx], y.iloc[test_idx]
    yb_f_tr = y_bin.iloc[train_idx]

    # Escalado dentro del fold — sin contaminacion
    sc_f = StandardScaler()
    X_f_tr_sc = sc_f.fit_transform(X_f_tr)
    X_f_te_sc  = sc_f.transform(X_f_te)

    # Etapa 1: clasificador
    clf_f = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf_f.fit(X_f_tr_sc, yb_f_tr)

    # Etapa 2: regresor solo en no-ceros del fold de train
    mask_f = yb_f_tr.values == 1
    reg_f  = Lasso(alpha=0.001, max_iter=10000, random_state=42)
    reg_f.fit(X_f_tr_sc[mask_f], y_f_tr.values[mask_f])

    # Prediccion combinada
    proba_f   = clf_f.predict_proba(X_f_te_sc)[:, 1]
    reg_pred_f = np.maximum(reg_f.predict(X_f_te_sc), 0)
    y_pred_f  = np.where(proba_f >= 0.5, reg_pred_f, 0.0)

    r2_hurdle_cv.append(r2_score(y_f_te, y_pred_f))

r2_hurdle_arr = np.array(r2_hurdle_cv)

# --- CV con Pipeline para Lasso solo (referencia directa) ---
pipe_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Lasso(alpha=0.001, max_iter=10000, random_state=42))
])
scores_lasso = cross_val_score(pipe_lasso, X, y, cv=kf, scoring='r2')

sep = '-'
print('=== CROSS-VALIDATION 5-fold — dataset completo (target: TUMOR) ===')
print(f'  {"Modelo":<38}  {"Media R2":>9}  {"Std":>7}  {"Min":>7}  {"Max":>7}')
print(f'  {sep*38}  {sep*9}  {sep*7}  {sep*7}  {sep*7}')
print(f'  {"Lasso       (alpha=0.001)":<38}  {scores_lasso.mean():>9.4f}  {scores_lasso.std():>7.4f}  {scores_lasso.min():>7.4f}  {scores_lasso.max():>7.4f}')
print(f'  {"Hurdle (Logistic + Lasso)":<38}  {r2_hurdle_arr.mean():>9.4f}  {r2_hurdle_arr.std():>7.4f}  {r2_hurdle_arr.min():>7.4f}  {r2_hurdle_arr.max():>7.4f}')
print()
print('Nota: TabPFN evaluado con 5 semillas (seccion 7b): media=0.0850 +/- 0.0161')
print('Referencia (un solo split seed=42): Lasso=0.0846 | Hurdle=0.0484 | TabPFN=~0.1001')
print('Referencia CV cruzada (03c, mismas features): Lasso media=0.0554 +/- 0.0282')